In [62]:
import pandas as pd

In [63]:
dt1 = pd.read_csv("data/processed/train1.csv")

In [64]:
dt1

,title,text,subject,date,label
0,WATCH SHOCKING DISPLAY of Muslim Intimidation ...,This is in-your-face taunting of our president...,politics,2017-06-04,1
1,Two Florida Lawmakers Just Voted Against Hurri...,Two Florida Republican lawmakers voted against...,News,2017-09-09,1
2,BOOM! HARRIS FAULKNER Blows Up The Russia Coll...,Fox News Harris Faulkner BLOWS UP the WHOLE R...,politics,2017-05-29,1
3,House of Representatives to try again to seek ...,WASHINGTON (Reuters) - The U.S. House of Repre...,politicsNews,2016-01-13,0
4,German citizen on trial in Turkey on political...,FRANKFURT (Reuters) - A 49-year-old woman has ...,worldnews,2017-09-07,0
...,...,...,...,...,...
23087,Hawaii judge halts Trump's new travel ban befo...,HONOLULU/NEW YORK (Reuters) - Just hours befor...,politicsNews,2017-03-15,0
23088,Turkey's Erdogan: Iraqi Kurds' decision not to...,ANKARA (Reuters) - Iraqi Kurdish leader Massou...,worldnews,2017-09-15,0
23089,Trump names Don McGahn as White House Counsel ...,"WASHINGTON/WEST PALM BEACH, Fla. (Reuters) - U...",politicsNews,2016-11-25,0
23090,A Van Struck Down Muslims In London,A horrifying incident took place in the early ...,News,2017-06-19,1


In [65]:
dt2 = pd.read_csv("data/processed/train2.csv")

In [66]:
dt2

,title,text,subject,date,label
0,Hanson's 'battler bus' takes the anti-immigran...,"TOWNSVILLE, Australia (Reuters) - It was early...",worldnews,2017-11-16,0
1,Lebanese army to get $120 million in U.S. aid,"BEIRUT (Reuters) - The United States, which wa...",worldnews,2017-12-13,0
2,Turkey's Erdogan says no problem with Russian ...,ISTANBUL (Reuters) - President Tayyip Erdogan ...,worldnews,2017-10-13,0
3,"Trump says he is 'very, very close' to making ...",WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,2017-10-23,0
4,Callista Gingrich becomes Trump's envoy to pop...,"VATICAN CITY (Reuters) - Callista Gingrich, wi...",politicsNews,2017-12-22,0
...,...,...,...,...,...
7810,A picture and its story: tear gas in Nairobi,"NAIROBI (Reuters) - In a dramatic picture, Ken...",worldnews,2017-10-13,0
7811,WATCH: INTOLERANT GAY COFFEE SHOP Owner Scream...,Remember the time when a gay couple destroyed ...,politics,2017-10-08,1
7812,JUST IN: Anti-Putin Banker Claims Firm Tied To...,One of the many targets of the opposition rese...,politics,2017-11-13,1
7813,Lebanese president calls Hariri's situation in...,BEIRUT (Reuters) - Lebanon s president said on...,worldnews,2017-11-11,0


In [67]:
import numpy as np

In [68]:
 df = load_and_merge("data/processed/train1.csv",
        "data/processed/train2.csv",
    )

2026-03-22 01:18:35,990 | INFO | train1: (23092, 6), train2: (7815, 6), combined: (30907, 6)


In [69]:
import json
import logging
from pathlib import Path
from datetime import datetime

In [70]:
# создаем директорию для вывода результатов
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
LOGGER = logging.getLogger("data_analysis")

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [71]:
def load_and_merge(train1_path: str, train2_path: str) -> pd.DataFrame:
    """Загружает два CSV и объединяет, сохраняя метку источника."""
    df1 = pd.read_csv(train1_path)
    df2 = pd.read_csv(train2_path)

    df1["source"] = "train1"
    df2["source"] = "train2"

    df = pd.concat([df1, df2], ignore_index=True)
    LOGGER.info(f"train1: {df1.shape}, train2: {df2.shape}, combined: {df.shape}")
    return df

In [72]:

def assess_data_quality(df: pd.DataFrame, save_path: Path = None) -> dict:
    if save_path is None:
        save_path = REPORTS_DIR / "data_quality.json"

    report = {}

    # Общая информация
    report["shape"] = {"n_rows": len(df), "n_cols": len(df.columns)}
    report["columns"] = list(df.columns)
    report["dtypes"] = {col: str(dtype) for col, dtype in df.dtypes.items()}

    # Пропуски
    missing_abs = df.isnull().sum()
    missing_pct = df.isnull().mean()
    report["missing"] = {
        col: {
            "count": int(missing_abs[col]),
            "percent": round(float(missing_pct[col]) * 100, 4),
        }
        for col in df.columns
    }
    report["total_missing_cells"] = int(missing_abs.sum())
    report["total_missing_percent"] = round(float(missing_pct.mean()) * 100, 4)

    # Дубликаты
    report["duplicates"] = {
        "full_row_duplicates": int(df.duplicated().sum()),
        "text_duplicates": int(df.duplicated(subset=["text"]).sum()) if "text" in df.columns else None,
    }

    # Целевая переменная
    if "label" in df.columns:
        vc = df["label"].value_counts()
        report["label_distribution"] = {
            str(k): int(v) for k, v in vc.items()
        }
        report["label_balance"] = round(float(df["label"].mean()), 4)

    # Текстовые статистики
    if "text" in df.columns:
        text_lens = df["text"].fillna("").str.len()
        word_counts = df["text"].fillna("").str.split().str.len()
        report["text_length_chars"] = _numeric_stats(text_lens)
        report["text_length_words"] = _numeric_stats(word_counts)
        report["empty_texts"] = int((text_lens == 0).sum())
        report["short_texts_under_50_chars"] = int((text_lens < 50).sum())

    if "title" in df.columns:
        title_lens = df["title"].fillna("").str.len()
        report["title_length_chars"] = _numeric_stats(title_lens)
        report["empty_titles"] = int((title_lens == 0).sum())

    # Категориальные распределения
    if "subject" in df.columns:
        report["subject_distribution"] = {
            str(k): int(v) for k, v in df["subject"].value_counts().items()
        }

    if "source" in df.columns:
        report["source_distribution"] = {
            str(k): int(v) for k, v in df["source"].value_counts().items()
        }

    # Временной диапазон
    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        valid_dates = dates.dropna()
        if len(valid_dates) > 0:
            report["date_range"] = {
                "min": str(valid_dates.min().date()),
                "max": str(valid_dates.max().date()),
                "n_invalid_dates": int(dates.isnull().sum()),
            }

    # Уникальность
    report["unique_counts"] = {
        col: int(df[col].nunique()) for col in df.columns
    }

    # Сохраняем
    report["created_at"] = datetime.now().isoformat()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    LOGGER.info(f"Data quality report saved: {save_path}")

    # Красивый вывод
    _print_quality_report(report)

    return report


def _numeric_stats(series: pd.Series) -> dict:
    return {
        "mean": round(float(series.mean()), 2),
        "std": round(float(series.std()), 2),
        "min": int(series.min()),
        "max": int(series.max()),
        "median": round(float(series.median()), 2),
        "q25": round(float(series.quantile(0.25)), 2),
        "q75": round(float(series.quantile(0.75)), 2),
    }


def _print_quality_report(report: dict):
    print("\n" + "=" * 60)
    print("  DATA QUALITY REPORT")
    print("=" * 60)

    print(f"\n Shape: {report['shape']['n_rows']} rows × {report['shape']['n_cols']} cols")

    # Пропуски
    print("\n Missing values:")
    has_missing = False
    for col, info in report["missing"].items():
        if info["count"] > 0:
            print(f"   {col}: {info['count']} ({info['percent']:.2f}%)")
            has_missing = True
    if not has_missing:
        print("   No missing values!")

    # Дубликаты
    dup = report["duplicates"]
    print(f"\n Duplicates:")
    print(f"   Full row: {dup['full_row_duplicates']}")
    if dup["text_duplicates"] is not None:
        print(f"   By text:  {dup['text_duplicates']}")

    # Фейк/правда
    if "label_distribution" in report:
        print(f"\n Label distribution:")
        for label, count in report["label_distribution"].items():
            print(f"   {label}: {count}")
        print(f"   Balance (mean label): {report['label_balance']}")

    # Текстовые статистики
    if "text_length_chars" in report:
        ts = report["text_length_chars"]
        ws = report["text_length_words"]
        print(f"\n Text length (chars): mean={ts['mean']}, median={ts['median']}, "
              f"min={ts['min']}, max={ts['max']}")
        print(f"   Text length (words): mean={ws['mean']}, median={ws['median']}, "
              f"min={ws['min']}, max={ws['max']}")
        print(f"   Empty texts: {report['empty_texts']}, "
              f"short (<50 chars): {report['short_texts_under_50_chars']}")

    #  Распределение по темам
    if "subject_distribution" in report:
        print(f"\n Subject distribution:")
        for subj, count in report["subject_distribution"].items():
            print(f"   {subj}: {count}")

    # Временной диапазон
    if "date_range" in report:
        dr = report["date_range"]
        print(f"\n Date range: {dr['min']} — {dr['max']}")
        if dr["n_invalid_dates"] > 0:
            print(f"   Invalid dates: {dr['n_invalid_dates']}")

    print("\n" + "=" * 60)

In [73]:
quality_report = assess_data_quality(df)

2026-03-22 01:18:37,580 | INFO | Data quality report saved: reports/data_quality.json



  DATA QUALITY REPORT

 Shape: 30907 rows × 6 cols

 Missing values:
   text: 1 (0.00%)

 Duplicates:
   Full row: 0
   By text:  0

 Label distribution:
   0: 16952
   1: 13955
   Balance (mean label): 0.4515

 Text length (chars): mean=2455.95, median=2230.0, min=0, max=51793
   Text length (words): mean=403.32, median=369.0, min=0, max=8135
   Empty texts: 1, short (<50 chars): 112

 Subject distribution:
   politicsNews: 8986
   worldnews: 7966
   News: 7230
   politics: 5138
   US_News: 629
   left-news: 545
   Government News: 413

 Date range: 2015-03-31 — 2018-02-19



In [74]:
import mlxtend

In [75]:
# Словарь агрессивных лов для бинарного признака
AGGRESSION_KEYWORDS = [
    "attack", "kill", "murder", "destroy", "threat", "bomb", "terror",
    "shoot", "assault", "violent", "war", "dead", "death", "weapon",
    "explod", "hostage", "massacre", "slaughter", "execution", "genocide",
    "rape", "torture", "riot", "arson", "stab", "hate", "fury", "rage",
]


def create_binary_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Бинарные признаки:
      - is_fake: label == 1
      - is_real: label == 0
      - has_trump: упоминание Trump / trump в text или title
      - has_aggression: наличие агрессивных слов в text
      - is_long_text: длина text > 75-го перцентиля
      - is_short_text: длина text < 25-го перцентиля
      - has_missing_date: дата отсутствует или невалидна
      - has_empty_title: заголовок пустой
      - subject_*: one-hot по subject
      - source_train1 / source_train2: из какого сплита
    """
    bdf = pd.DataFrame(index=df.index)

    # Label 
    bdf["is_fake"] = (df["label"] == 1).astype(bool)
    bdf["is_real"] = (df["label"] == 0).astype(bool)

    # Упоминание Трампа
    text_lower = df["text"].fillna("").str.lower()
    title_lower = df["title"].fillna("").str.lower()
    bdf["has_trump"] = (
        text_lower.str.contains("donald trump", na=False)
        | title_lower.str.contains("donald trump", na=False)
    )

    # Агрессия
    pattern = "|".join(AGGRESSION_KEYWORDS)
    bdf["has_aggression"] = text_lower.str.contains(pattern, na=False)

    # Длина текста
    text_len = df["text"].fillna("").str.len()
    q25 = text_len.quantile(0.25)
    q75 = text_len.quantile(0.75)
    bdf["is_short_text"] = (text_len < q25)
    bdf["is_long_text"] = (text_len > q75)
    bdf["is_medium_text"] = (~bdf["is_short_text"]) & (~bdf["is_long_text"])

    # Пропуски
    dates = pd.to_datetime(df["date"], errors="coerce")
    bdf["has_missing_date"] = dates.isna()
    bdf["has_empty_title"] = df["title"].fillna("").str.strip().eq("")

    # Распарсим темы на несколько столбцов
    if "subject" in df.columns:
        for subj in df["subject"].dropna().unique():
            safe_name = subj.strip().replace(" ", "_").replace("-", "_").lower()
            bdf[f"subject_{safe_name}"] = (df["subject"] == subj)

    # Источник
    if "source" in df.columns:
        bdf["source_train1"] = (df["source"] == "train1")
        bdf["source_train2"] = (df["source"] == "train2")

    # Все колонки bool
    bdf = bdf.astype(bool)

    LOGGER.info(f"Created {len(bdf.columns)} binary features: {list(bdf.columns)}")
    return bdf


def mine_association_rules(
    df: pd.DataFrame,
    min_support: float = 0.05,
    min_confidence: float = 0.5,
    min_lift: float = 1.2,
    save_path: Path = None,
) -> pd.DataFrame:
    """
    Применяет FP-Growth для поиска частых наборов и генерирует ассоциативные правила.
    Выбирает не менее 5 правил, полезных для проверки корректности данных
    или обогащения признакового пространства.
    """
    from mlxtend.frequent_patterns import fpgrowth, association_rules

    if save_path is None:
        save_path = REPORTS_DIR / "association_rules.json"

    # Бинаризация
    bdf = create_binary_features(df)

    LOGGER.info(f"Running FP-Growth: min_support={min_support}")
    frequent_itemsets = fpgrowth(bdf, min_support=min_support, use_colnames=True)
    LOGGER.info(f"Found {len(frequent_itemsets)} frequent itemsets")

    if len(frequent_itemsets) == 0:
        LOGGER.warning("No frequent itemsets found. Try lowering min_support.")
        return pd.DataFrame()

    # Генерация правил
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=min_confidence,
        num_itemsets=len(frequent_itemsets),
    )
    LOGGER.info(f"Generated {len(rules)} rules (confidence >= {min_confidence})")

    # Фильтруем по lift
    rules = rules[rules["lift"] >= min_lift].copy()
    LOGGER.info(f"After lift >= {min_lift}: {len(rules)} rules")

    if len(rules) == 0:
        LOGGER.warning("No rules passed filters. Try lowering thresholds.")
        return pd.DataFrame()

    # Конвертируем frozenset в список для красивого вывода
    rules["antecedents_list"] = rules["antecedents"].apply(sorted)
    rules["consequents_list"] = rules["consequents"].apply(sorted)

    # Сортируем по lift и берём топ
    rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

    # Выбираем 5+ правил разного типа для проверки корректности / обогащения
    selected = _select_interesting_rules(rules)

    # Сохраняем
    report = {
        "total_frequent_itemsets": len(frequent_itemsets),
        "total_rules": len(rules),
        "params": {
            "min_support": min_support,
            "min_confidence": min_confidence,
            "min_lift": min_lift,
        },
        "selected_rules": [],
        "all_rules_top30": [],
    }

    for _, row in selected.iterrows():
        report["selected_rules"].append(_rule_to_dict(row))

    for _, row in rules.head(30).iterrows():
        report["all_rules_top30"].append(_rule_to_dict(row))

    report["created_at"] = datetime.now().isoformat()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    LOGGER.info(f"Association rules report saved: {save_path}")

    # Вывод
    _print_rules(selected, title="SELECTED RULES (for data quality / feature enrichment)")
    _print_rules(rules.head(15), title="TOP-15 RULES BY LIFT")

    return rules


def _rule_to_dict(row) -> dict:
    return {
        "antecedents": row["antecedents_list"],
        "consequents": row["consequents_list"],
        "support": round(float(row["support"]), 4),
        "confidence": round(float(row["confidence"]), 4),
        "lift": round(float(row["lift"]), 4),
        "conviction": round(float(row["conviction"]), 4) if np.isfinite(row["conviction"]) else "inf",
    }


def _select_interesting_rules(rules: pd.DataFrame, min_rules: int = 5) -> pd.DataFrame:
    """
    берём правила с разными consequents, приоритизируя:
      1. Правила с is_fake / is_real в consequent (валидация label)
      2. Правила с has_trump / has_aggression (валидация контента)
      3. Правила с subject_* (валидация категорий)
      4. Правила с наибольшим lift (неожиданные зависимости)
    """
    selected_indices = set()

    # Приоритетные consequents
    priority_consequents = [
        ["is_fake"], ["is_real"],
        ["has_trump"], ["has_aggression"],
    ]

    for target in priority_consequents:
        target_set = frozenset(target)
        mask = rules["consequents"] == target_set
        candidates = rules[mask]
        if len(candidates) > 0:
            # берём правило с наибольшим lift
            best_idx = candidates["lift"].idxmax()
            selected_indices.add(best_idx)

    # Добавляем правила с subject в consequent
    subject_rules = rules[
        rules["consequents_list"].apply(
            lambda x: any(c.startswith("subject_") for c in x)
        )
    ]
    if len(subject_rules) > 0:
        best_idx = subject_rules["lift"].idxmax()
        selected_indices.add(best_idx)

    # Добиваем до min_rules из топа по lift
    for idx in rules.index:
        if len(selected_indices) >= min_rules:
            break
        selected_indices.add(idx)

    selected = rules.loc[sorted(selected_indices)].copy()
    LOGGER.info(f"Selected {len(selected)} interesting rules")
    return selected


def _print_rules(rules: pd.DataFrame, title: str = "ASSOCIATION RULES"):
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")

    if len(rules) == 0:
        print("  No rules found.")
        print(f"{'=' * 70}")
        return

    for i, (_, row) in enumerate(rules.iterrows(), 1):
        ant = ", ".join(row["antecedents_list"])
        cons = ", ".join(row["consequents_list"])
        print(f"\n  Rule {i}:")
        print(f"    {ant} : {cons}")
        print(f"    support={row['support']:.4f}  "
              f"confidence={row['confidence']:.4f}  "
              f"lift={row['lift']:.4f}")

    print(f"\n{'=' * 70}")


In [76]:
rules = mine_association_rules(df)

2026-03-22 01:18:40,014 | INFO | Created 18 binary features: ['is_fake', 'is_real', 'has_trump', 'has_aggression', 'is_short_text', 'is_long_text', 'is_medium_text', 'has_missing_date', 'has_empty_title', 'subject_politics', 'subject_news', 'subject_politicsnews', 'subject_worldnews', 'subject_left_news', 'subject_government_news', 'subject_us_news', 'source_train1', 'source_train2']
2026-03-22 01:18:40,020 | INFO | Running FP-Growth: min_support=0.05
2026-03-22 01:18:40,133 | INFO | Found 226 frequent itemsets
2026-03-22 01:18:40,141 | INFO | Generated 736 rules (confidence >= 0.5)
2026-03-22 01:18:40,143 | INFO | After lift >= 1.2: 566 rules
2026-03-22 01:18:40,148 | INFO | Selected 5 interesting rules
2026-03-22 01:18:40,152 | INFO | Association rules report saved: reports/association_rules.json



  SELECTED RULES (for data quality / feature enrichment)

  Rule 1:
    is_fake, is_short_text : source_train1, subject_politics
    support=0.0580  confidence=0.7331  lift=4.9702

  Rule 2:
    subject_news : is_fake
    support=0.2339  confidence=1.0000  lift=2.2148

  Rule 3:
    has_aggression, is_long_text, is_real, subject_politicsnews : has_trump
    support=0.0671  confidence=0.8452  lift=1.9825

  Rule 4:
    is_long_text, subject_politicsnews : is_real
    support=0.0875  confidence=1.0000  lift=1.8232

  Rule 5:
    is_long_text, is_real, subject_worldnews : has_aggression
    support=0.0562  confidence=0.9554  lift=1.3038


  TOP-15 RULES BY LIFT

  Rule 1:
    is_long_text, subject_politicsnews : has_aggression, has_trump, is_real, source_train1
    support=0.0590  confidence=0.6736  lift=5.2346

  Rule 2:
    is_fake, is_short_text : source_train1, subject_politics
    support=0.0580  confidence=0.7331  lift=4.9702

  Rule 3:
    is_long_text, source_train1, subject_poli

In [77]:
# Пороги по умолчанию — можно переопределить при вызове
DEFAULT_CLEANING_THRESHOLDS = {
    "min_text_length_chars": 50,      # текст короче — мусор / заглушка
    "max_text_length_chars": 50_000,  # текст длиннее — возможный дубль / парсинг-артефакт
    "min_title_length_chars": 3,      # заголовок короче — пустышка
    "max_duplicate_text_keep": 1,     # оставляем только первый экземпляр дубликата по text
    "drop_missing_text": True,        # удаляем строки с пустым text
    "drop_missing_label": True,       # удаляем строки без label
    "drop_invalid_dates": False,      # НЕ удаляем по умолчанию (дата не критична для модели)
    "allowed_labels": [0, 1],         # допустимые значения label
}


def clean_data(
    df: pd.DataFrame,
    thresholds: dict = None,
    save_report_path: Path = None,
) -> pd.DataFrame:
    """
      1. Удаление строк с пропущенным text (если включено)
      2. Удаление строк с пропущенным / невалидным label
      3. Фильтр по длине текста (min / max символов)
      4. Фильтр по длине заголовка
      5. Удаление полных дубликатов по text
      6. Удаление строк с невалидными датами (опционально)
    """
    if thresholds is None:
        thresholds = DEFAULT_CLEANING_THRESHOLDS
    if save_report_path is None:
        save_report_path = REPORTS_DIR / "cleaning_report.json"

    t = {**DEFAULT_CLEANING_THRESHOLDS, **thresholds}
    n_before = len(df)
    df = df.copy()

    cleaning_log = []  # (step_name, n_removed)

    def _log_step(name: str, df_before_len: int, df_after: pd.DataFrame) -> pd.DataFrame:
        removed = df_before_len - len(df_after)
        cleaning_log.append({"step": name, "removed": removed, "remaining": len(df_after)})
        if removed > 0:
            LOGGER.info(f"  [{name}] removed {removed} rows → {len(df_after)} remaining")
        return df_after

    LOGGER.info(f"Starting data cleaning: {n_before} rows")

    # Пропущенный text
    if t["drop_missing_text"] and "text" in df.columns:
        n = len(df)
        df = df.dropna(subset=["text"])
        df = df[df["text"].astype(str).str.strip() != ""]
        df = _log_step("drop_missing_text", n, df)

    # Пропущенный / невалидный label
    if t["drop_missing_label"] and "label" in df.columns:
        n = len(df)
        df = df.dropna(subset=["label"])
        df["label"] = pd.to_numeric(df["label"], errors="coerce")
        df = df.dropna(subset=["label"])
        df["label"] = df["label"].astype(int)
        if t["allowed_labels"]:
            df = df[df["label"].isin(t["allowed_labels"])]
        df = _log_step("drop_invalid_label", n, df)

    # Фильтр по длине текста
    if "text" in df.columns:
        text_len = df["text"].str.len()

        n = len(df)
        df = df[text_len >= t["min_text_length_chars"]]
        df = _log_step(f"min_text_length >= {t['min_text_length_chars']}", n, df)

        text_len = df["text"].str.len()  # пересчитываем после предыдущего шага
        n = len(df)
        df = df[text_len <= t["max_text_length_chars"]]
        df = _log_step(f"max_text_length <= {t['max_text_length_chars']}", n, df)

    # Фильтр по длине заголовка
    if "title" in df.columns:
        # Не удаляем строки без заголовка — просто заполняем пустой строкой
        df["title"] = df["title"].fillna("")
        title_len = df["title"].str.len()
        n = len(df)
        # Удаляем только если title существует, но слишком короткий (не пустой — пустой ок)
        mask = (title_len >= t["min_title_length_chars"]) | (title_len == 0)
        df = df[mask]
        df = _log_step(f"min_title_length >= {t['min_title_length_chars']}", n, df)

    # Дубликаты по text
    n = len(df)
    df = df.drop_duplicates(subset=["text"], keep="first")
    df = _log_step("drop_text_duplicates", n, df)

    # Невалидные даты (опционально)
    if t["drop_invalid_dates"] and "date" in df.columns:
        n = len(df)
        dates = pd.to_datetime(df["date"], errors="coerce")
        df = df[dates.notna()]
        df = _log_step("drop_invalid_dates", n, df)

    df = df.reset_index(drop=True)
    n_after = len(df)

    report = {
        "rows_before": n_before,
        "rows_after": n_after,
        "total_removed": n_before - n_after,
        "removal_percent": round((n_before - n_after) / n_before * 100, 2) if n_before > 0 else 0,
        "thresholds": t,
        "steps": cleaning_log,
        "created_at": datetime.now().isoformat(),
    }

    save_report_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2, default=str)
    LOGGER.info(f"Cleaning report saved: {save_report_path}")

    _print_cleaning_report(report)
    return df


def _print_cleaning_report(report: dict):
    print(f"\n{'=' * 60}")
    print("  DATA CLEANING REPORT")
    print(f"{'=' * 60}")
    print(f"\n  Before: {report['rows_before']}   After: {report['rows_after']}  "
          f"(removed {report['total_removed']}, {report['removal_percent']}%)")
    print(f"\n  Steps:")
    for step in report["steps"]:
        marker = "bad" if step["removed"] > 0 else "good"
        print(f"    {marker} {step['step']}: delete {step['removed']}  (after: {step['remaining']})")
    print(f"\n{'=' * 60}")

  

In [78]:
 df = clean_data(df)

2026-03-22 01:18:40,186 | INFO | Starting data cleaning: 30907 rows
2026-03-22 01:18:40,206 | INFO |   [drop_missing_text] removed 1 rows → 30906 remaining
2026-03-22 01:18:40,218 | INFO |   [min_text_length >= 50] removed 111 rows → 30795 remaining
2026-03-22 01:18:40,226 | INFO |   [max_text_length <= 50000] removed 1 rows → 30794 remaining
2026-03-22 01:18:40,234 | INFO | Cleaning report saved: reports/cleaning_report.json



  DATA CLEANING REPORT

  Before: 30907   After: 30794  (removed 113, 0.37%)

  Steps:
    bad drop_missing_text: delete 1  (after: 30906)
    good drop_invalid_label: delete 0  (after: 30906)
    bad min_text_length >= 50: delete 111  (after: 30795)
    bad max_text_length <= 50000: delete 1  (after: 30794)
    good min_title_length >= 3: delete 0  (after: 30794)
    good drop_text_duplicates: delete 0  (after: 30794)



In [87]:
import sweetviz as sv

In [94]:
def run_automatic_eda(
    df: pd.DataFrame,
    output_dir: Path = None,
    sample_size: int = 5000,
):
    """
    Запускает автоматический EDA:
      - sweetviz: сравнительный HTML-отчёт fake vs real
      - matplotlib: набор графиков в PDF (распределения, корреляции, по классам)

    Аргументы:
        df: DataFrame после очистки
        output_dir: директория для отчётов
        sample_size: максимальный размер выборки (0 = без семплирования)
    """
    if output_dir is None:
        output_dir = REPORTS_DIR
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    eda_df = _prepare_eda_dataframe(df)

    if sample_size and len(eda_df) > sample_size:
        LOGGER.info(f"Sampling {sample_size} rows from {len(eda_df)} for EDA")
        eda_df = eda_df.sample(n=sample_size, random_state=42)

    # Sweetviz — сравнительный HTML-отчёт
    _run_sweetviz(eda_df, output_dir)

    # Matplotlib — PDF с графиками
    _run_matplotlib_eda(eda_df, output_dir)


def _prepare_eda_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Подготавливает DataFrame для EDA:
    убираем сырой text (слишком длинный), добавляем числовые агрегаты.
    """
    eda = df.copy()

    if "text" in eda.columns:
        eda["text_len_chars"] = eda["text"].fillna("").str.len()
        eda["text_len_words"] = eda["text"].fillna("").str.split().str.len()
        eda["text_avg_word_len"] = (
            eda["text_len_chars"] / eda["text_len_words"].replace(0, np.nan)
        ).round(2)
        eda["text_n_sentences"] = eda["text"].fillna("").str.count(r"[.!?]+")
        eda["text_n_exclamations"] = eda["text"].fillna("").str.count("!")
        eda["text_n_questions"] = eda["text"].fillna("").str.count(r"\?")
        eda["text_n_caps_words"] = eda["text"].fillna("").str.findall(r"\b[A-Z]{2,}\b").str.len()
        eda["text_upper_ratio"] = (
            eda["text"].fillna("").apply(lambda t: sum(c.isupper() for c in t))
            / eda["text_len_chars"].replace(0, np.nan)
        ).round(4)

    if "title" in eda.columns:
        eda["title_len_chars"] = eda["title"].fillna("").str.len()
        eda["title_len_words"] = eda["title"].fillna("").str.split().str.len()

    text_lower = eda["text"].fillna("").str.lower() if "text" in eda.columns else pd.Series("", index=eda.index)
    title_lower = eda["title"].fillna("").str.lower() if "title" in eda.columns else pd.Series("", index=eda.index)
    eda["has_trump"] = (
        text_lower.str.contains("donald trump", na=False)
        | title_lower.str.contains("donald trump", na=False)
    ).astype(int)

    pattern = "|".join(AGGRESSION_KEYWORDS)
    eda["has_aggression"] = text_lower.str.contains(pattern, na=False).astype(int)

    eda = eda.drop(columns=["text", "title"], errors="ignore")
    eda = eda.drop(columns=["arrival_time"], errors="ignore")

    return eda


def _run_sweetviz(eda_df: pd.DataFrame, output_dir: Path):
    # Генерирует sweetviz HTML отчёт со сравнением fake vs real
    try:
        if not hasattr(np, "VisibleDeprecationWarning"):
            np.VisibleDeprecationWarning = FutureWarning
        LOGGER.info("Generating sweetviz report (fake vs real comparison)...")

        if "label" in eda_df.columns:
            fake_df = eda_df[eda_df["label"] == 1]
            real_df = eda_df[eda_df["label"] == 0]
            report = sv.compare(
                [fake_df, "Fake News"],
                [real_df, "Real News"],
                target_feat="label",
            )
        else:
            report = sv.analyze(eda_df)

        report_path = output_dir / "eda_sweetviz.html"
        report.show_html(filepath=str(report_path), open_browser=False)
        LOGGER.info(f"sweetviz report saved: {report_path}")

    except Exception as e:
        LOGGER.error(f"sweetviz failed: {e}")


def _run_matplotlib_eda(eda_df: pd.DataFrame, output_dir: Path):
    """
    Генерирует многостраничный PDF с EDA-графиками:
      1. Распределение целевой переменной (label)
      2. Распределение по subject и source
      3. Гистограммы числовых признаков
      4. Гистограммы числовых признаков по классам (fake/real)
      5. Корреляционная матрица
      6. Boxplot числовых фичей по label
      7. Бинарные признаки: частоты по label
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    import seaborn as sns

    report_path = output_dir / "eda_plots.pdf"
    LOGGER.info("Generating matplotlib EDA report...")

    numeric_cols = eda_df.select_dtypes(include=[np.number]).columns.tolist()
    # Убираем label из числовых для графиков распределений
    feature_numeric = [c for c in numeric_cols if c != "label"]
    categorical_cols = [c for c in ["subject", "source", "date"] if c in eda_df.columns]

    with PdfPages(str(report_path)) as pdf:

        # Распределение по Label 
        if "label" in eda_df.columns:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            fig.suptitle("Target Variable Distribution", fontsize=14, fontweight="bold")

            vc = eda_df["label"].value_counts().sort_index()
            labels_map = {0: "Real", 1: "Fake"}
            colors = ["#2ecc71", "#e74c3c"]

            axes[0].bar([labels_map.get(k, str(k)) for k in vc.index], vc.values, color=colors)
            axes[0].set_title("Count")
            axes[0].set_ylabel("Number of samples")
            for i, v in enumerate(vc.values):
                axes[0].text(i, v + len(eda_df) * 0.01, str(v), ha="center", fontweight="bold")

            axes[1].pie(vc.values, labels=[labels_map.get(k, str(k)) for k in vc.index],
                        autopct="%1.1f%%", colors=colors, startangle=90)
            axes[1].set_title("Proportion")

            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

        # Распределение по subject и source
        cat_to_plot = [c for c in ["subject", "source"] if c in eda_df.columns]
        if cat_to_plot:
            fig, axes = plt.subplots(1, len(cat_to_plot), figsize=(7 * len(cat_to_plot), 5))
            if len(cat_to_plot) == 1:
                axes = [axes]
            fig.suptitle("Categorical Feature Distributions", fontsize=14, fontweight="bold")

            for ax, col in zip(axes, cat_to_plot):
                if "label" in eda_df.columns:
                    ct = pd.crosstab(eda_df[col], eda_df["label"])
                    ct.columns = ["Real", "Fake"]
                    ct.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white")
                    ax.legend(title="Label")
                else:
                    eda_df[col].value_counts().plot(kind="bar", ax=ax, color="#3498db", edgecolor="white")
                ax.set_title(col)
                ax.set_ylabel("Count")
                ax.tick_params(axis="x", rotation=45)

            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

        # Гистограммы числовых признаков
        if feature_numeric:
            n_feats = len(feature_numeric)
            cols_per_page = 3
            rows_per_page = 4
            per_page = cols_per_page * rows_per_page

            for page_start in range(0, n_feats, per_page):
                page_feats = feature_numeric[page_start:page_start + per_page]
                n_plots = len(page_feats)
                n_rows = (n_plots + cols_per_page - 1) // cols_per_page
                fig, axes = plt.subplots(n_rows, cols_per_page, figsize=(15, 4 * n_rows))
                fig.suptitle("Numeric Feature Distributions", fontsize=14, fontweight="bold")
                axes_flat = axes.flatten() if n_plots > 1 else [axes]

                for i, col in enumerate(page_feats):
                    ax = axes_flat[i]
                    if "label" in eda_df.columns:
                        for label_val, color, name in [(0, "#2ecc71", "Real"), (1, "#e74c3c", "Fake")]:
                            subset = eda_df.loc[eda_df["label"] == label_val, col].dropna()
                            ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, edgecolor="white")
                        ax.legend(fontsize=8)
                    else:
                        ax.hist(eda_df[col].dropna(), bins=30, color="#3498db", edgecolor="white", alpha=0.7)
                    ax.set_title(col, fontsize=10)
                    ax.tick_params(labelsize=8)

                for i in range(n_plots, len(axes_flat)):
                    axes_flat[i].set_visible(False)

                plt.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)

        # Корреляционная матрица
        if len(feature_numeric) >= 2:
            fig, ax = plt.subplots(figsize=(14, 11))
            corr_cols = feature_numeric
            if "label" in eda_df.columns:
                corr_cols = ["label"] + feature_numeric
            corr = eda_df[corr_cols].corr()
            mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
            sns.heatmap(
                corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, ax=ax, annot_kws={"size": 7}, linewidths=0.5,
                vmin=-1, vmax=1,
            )
            ax.set_title("Correlation Matrix", fontsize=14, fontweight="bold")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

        # Boxplot числовых фичей по label (топ по информативности)
        if "label" in eda_df.columns and feature_numeric:
            # Выбираем наиболее информативные фичи (топ по разнице медиан)
            median_diff = {}
            for col in feature_numeric:
                m0 = eda_df.loc[eda_df["label"] == 0, col].median()
                m1 = eda_df.loc[eda_df["label"] == 1, col].median()
                std_val = eda_df[col].std()
                if std_val and std_val > 0:
                    median_diff[col] = abs(m1 - m0) / std_val
                else:
                    median_diff[col] = 0.0

            top_feats = sorted(median_diff, key=median_diff.get, reverse=True)[:12]

            if top_feats:
                n_rows = (len(top_feats) + 2) // 3
                fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
                fig.suptitle("Top Features: Boxplot by Label", fontsize=14, fontweight="bold")
                axes_flat = axes.flatten()

                for i, col in enumerate(top_feats):
                    ax = axes_flat[i]
                    data_real = eda_df.loc[eda_df["label"] == 0, col].dropna()
                    data_fake = eda_df.loc[eda_df["label"] == 1, col].dropna()
                    bp = ax.boxplot(
                        [data_real, data_fake],
                        labels=["Real", "Fake"],
                        patch_artist=True,
                        widths=0.6,
                    )
                    bp["boxes"][0].set_facecolor("#2ecc71")
                    bp["boxes"][1].set_facecolor("#e74c3c")
                    ax.set_title(f"{col}\n(Δmedian/σ={median_diff[col]:.2f})", fontsize=9)
                    ax.tick_params(labelsize=8)

                for i in range(len(top_feats), len(axes_flat)):
                    axes_flat[i].set_visible(False)

                plt.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)

        # Бинарные признаки: частоты по label
        binary_cols = [c for c in feature_numeric if eda_df[c].dropna().isin([0, 1]).all() and c != "label"]
        if binary_cols and "label" in eda_df.columns:
            rates = []
            for col in binary_cols:
                rate_real = eda_df.loc[eda_df["label"] == 0, col].mean()
                rate_fake = eda_df.loc[eda_df["label"] == 1, col].mean()
                rates.append({"feature": col, "Real": rate_real, "Fake": rate_fake})
            rates_df = pd.DataFrame(rates).set_index("feature")

            fig, ax = plt.subplots(figsize=(12, max(5, len(binary_cols) * 0.4)))
            rates_df.plot(kind="barh", ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white")
            ax.set_title("Binary Feature Rates by Label", fontsize=14, fontweight="bold")
            ax.set_xlabel("Rate (proportion of 1s)")
            ax.legend(title="Label")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    LOGGER.info(f"Matplotlib EDA report saved: {report_path}")

In [95]:
run_automatic_eda(df)

2026-03-22 02:15:46,366 | INFO | Sampling 5000 rows from 30794 for EDA
2026-03-22 02:15:46,371 | INFO | Generating sweetviz report (fake vs real comparison)...
Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)
2026-03-22 02:15:52,874 | INFO | sweetviz report saved: reports/eda_sweetviz.html
2026-03-22 02:15:52,876 | INFO | Generating matplotlib EDA report...
/tmp/ipykernel_16991/1801632230.py:164: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  ct.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white")
/tmp/ipykernel_16991/1801632230.py:164: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies u

Report reports/eda_sweetviz.html was generated.


/tmp/ipykernel_16991/1801632230.py:253: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_16991/1801632230.py:253: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_16991/1801632230.py:253: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_16991/1801632230.py:253: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_16991/1801632230.py:253: MatplotlibDeprecationWarning: The 'labels' param

In [97]:

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Группы признаков:
      1. Текстовые статистики (длина, кол-во слов, предложений, ...)
      2. Стилистические маркеры (caps ratio, пунктуация, ...)
      3. Контентные бинарные (has_trump, has_aggression)
      4. Лексическое разнообразие (type-token ratio)
      5. Временные признаки (day_of_week, month, ...)
      6. Subject-based (уже категориальный)
    """
    df = df.copy()
    LOGGER.info(f"Feature engineering: starting with {len(df.columns)} columns")

    text = df["text"].fillna("")
    title = df["title"].fillna("") if "title" in df.columns else pd.Series("", index=df.index)

    # Текстовые статистики
    df["text_len_chars"] = text.str.len()
    df["text_len_words"] = text.str.split().str.len().fillna(0).astype(int)
    df["text_n_sentences"] = text.str.count(r"[.!?]+")
    df["text_avg_word_len"] = (
        df["text_len_chars"] / df["text_len_words"].replace(0, np.nan)
    ).round(2)
    df["text_avg_sentence_len"] = (
        df["text_len_words"] / df["text_n_sentences"].replace(0, np.nan)
    ).round(2)

    df["title_len_chars"] = title.str.len()
    df["title_len_words"] = title.str.split().str.len().fillna(0).astype(int)

    # Отношение длины заголовка к длине текста
    df["title_to_text_ratio"] = (
        df["title_len_chars"] / df["text_len_chars"].replace(0, np.nan)
    ).round(4)

    # Стилистические маркеры
    # Доля заглавных букв
    df["upper_char_count"] = text.apply(lambda t: sum(c.isupper() for c in t))
    df["upper_ratio"] = (
        df["upper_char_count"] / df["text_len_chars"].replace(0, np.nan)
    ).round(4)

    # Слова целиком в верхнем регистре (CAPS LOCK)
    df["caps_word_count"] = text.str.findall(r"\b[A-Z]{2,}\b").str.len()
    df["caps_word_ratio"] = (
        df["caps_word_count"] / df["text_len_words"].replace(0, np.nan)
    ).round(4)

    # Пунктуация
    df["exclamation_count"] = text.str.count("!")
    df["question_count"] = text.str.count(r"\?")
    df["ellipsis_count"] = text.str.count(r"\.\.\.")
    df["quote_count"] = text.str.count(r'["\u201c\u201d\u2018\u2019]')

    # Общее кол-во знаков пунктуации
    df["punct_count"] = text.str.count(r"[^\w\s]")
    df["punct_ratio"] = (
        df["punct_count"] / df["text_len_chars"].replace(0, np.nan)
    ).round(4)

    # Контентные бинарные признаки
    text_lower = text.str.lower()
    title_lower = title.str.lower()

    df["has_trump"] = (
        text_lower.str.contains("donald trump", na=False)
        | title_lower.str.contains("donald trump", na=False)
    ).astype(int)

    aggression_pattern = "|".join(AGGRESSION_KEYWORDS)
    df["has_aggression"] = text_lower.str.contains(aggression_pattern, na=False).astype(int)

    # Другие полезные контентные маркеры
    df["has_url"] = text.str.contains(r"https?://", na=False).astype(int)
    df["has_email"] = text.str.contains(r"\S+@\S+\.\S+", na=False).astype(int)
    df["has_number"] = text.str.contains(r"\d+", na=False).astype(int)
    df["number_count"] = text.str.findall(r"\b\d+\b").str.len()

    # Маркеры «кликбейтности»
    df["has_breaking"] = text_lower.str.contains(r"\bbreaking\b", na=False).astype(int)
    df["has_exclusive"] = text_lower.str.contains(r"\bexclusive\b", na=False).astype(int)
    df["has_shocking"] = text_lower.str.contains(
        r"\b(?:shock|unbelievable|stunning|bombshell|jaw.?dropping)\b", na=False
    ).astype(int)

    # Лексическое разнообразие
    def _type_token_ratio(t: str) -> float:
        words = t.lower().split()
        if len(words) == 0:
            return 0.0
        return len(set(words)) / len(words)

    df["type_token_ratio"] = text.apply(_type_token_ratio).round(4)

    # слова, встречающиеся ровно 1 раз
    def _hapax_ratio(t: str) -> float:
        words = t.lower().split()
        if len(words) == 0:
            return 0.0
        from collections import Counter
        freq = Counter(words)
        hapax = sum(1 for w, c in freq.items() if c == 1)
        return hapax / len(words)

    df["hapax_ratio"] = text.apply(_hapax_ratio).round(4)

    # Временные признаки
    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        df["day_of_week"] = dates.dt.dayofweek       # 0=Mon, 6=Sun
        df["month"] = dates.dt.month
        df["year"] = dates.dt.year
        df["is_weekend"] = dates.dt.dayofweek.isin([5, 6]).astype(int)

    new_cols = [c for c in df.columns if c not in ["text", "title", "date", "label", "subject", "source"]]
    LOGGER.info(f"Feature engineering: added {len(new_cols)} features → {len(df.columns)} total columns")
    LOGGER.info(f"New features: {new_cols}")

    return df


def _print_feature_summary(df: pd.DataFrame):
    #Выводит краткую сводку по новым признакам
    feature_cols = [c for c in df.columns if c not in ["text", "title", "date", "label", "subject", "source"]]

    print(f"\n{'=' * 60}")
    print("  FEATURE ENGINEERING SUMMARY")
    print(f"{'=' * 60}")
    print(f"\n  Total features: {len(feature_cols)}")

    # Группируем по типу
    numeric_feats = [c for c in feature_cols if df[c].dtype in ["float64", "float32", "int64", "int32"]]
    binary_feats = [c for c in numeric_feats if df[c].isin([0, 1]).all()]
    continuous_feats = [c for c in numeric_feats if c not in binary_feats]

    print(f"  Binary features ({len(binary_feats)}): {binary_feats}")
    print(f"  Continuous features ({len(continuous_feats)}): {continuous_feats}")

    # Статистики по бинарным
    if binary_feats:
        print(f"\n  Binary feature rates:")
        for feat in binary_feats:
            rate = df[feat].mean()
            print(f"    {feat}: {rate:.2%}")

    print(f"\n{'=' * 60}")

In [98]:
df = add_features(df)
_print_feature_summary(df)

2026-03-22 02:17:30,693 | INFO | Feature engineering: starting with 6 columns
2026-03-22 02:17:55,664 | INFO | Feature engineering: added 33 features → 39 total columns
2026-03-22 02:17:55,665 | INFO | New features: ['text_len_chars', 'text_len_words', 'text_n_sentences', 'text_avg_word_len', 'text_avg_sentence_len', 'title_len_chars', 'title_len_words', 'title_to_text_ratio', 'upper_char_count', 'upper_ratio', 'caps_word_count', 'caps_word_ratio', 'exclamation_count', 'question_count', 'ellipsis_count', 'quote_count', 'punct_count', 'punct_ratio', 'has_trump', 'has_aggression', 'has_url', 'has_email', 'has_number', 'number_count', 'has_breaking', 'has_exclusive', 'has_shocking', 'type_token_ratio', 'hapax_ratio', 'day_of_week', 'month', 'year', 'is_weekend']



  FEATURE ENGINEERING SUMMARY

  Total features: 33
  Binary features (9): ['has_trump', 'has_aggression', 'has_url', 'has_email', 'has_number', 'has_breaking', 'has_exclusive', 'has_shocking', 'is_weekend']
  Continuous features (24): ['text_len_chars', 'text_len_words', 'text_n_sentences', 'text_avg_word_len', 'text_avg_sentence_len', 'title_len_chars', 'title_len_words', 'title_to_text_ratio', 'upper_char_count', 'upper_ratio', 'caps_word_count', 'caps_word_ratio', 'exclamation_count', 'question_count', 'ellipsis_count', 'quote_count', 'punct_count', 'punct_ratio', 'number_count', 'type_token_ratio', 'hapax_ratio', 'day_of_week', 'month', 'year']

  Binary feature rates:
    has_trump: 42.77%
    has_aggression: 73.53%
    has_url: 6.37%
    has_email: 0.09%
    has_number: 81.42%
    has_breaking: 2.18%
    has_exclusive: 0.59%
    has_shocking: 2.57%
    is_weekend: 18.45%



In [101]:
def generate_quality_report(
    df: pd.DataFrame,
    quality_report: dict = None,
    cleaning_report_path: Path = None,
    output_path: Path = None,
) -> Path:
    # Генерирует сводный HTML-отчёт о качестве данных
    if output_path is None:
        output_path = REPORTS_DIR / "data_quality_report.html"
    if cleaning_report_path is None:
        cleaning_report_path = REPORTS_DIR / "cleaning_report.json"

    # Загружаем cleaning report если есть
    cleaning_info = {}
    if cleaning_report_path.exists():
        with open(cleaning_report_path, "r", encoding="utf-8") as f:
            cleaning_info = json.load(f)

    # Случай, когда quality_report не передан
    if quality_report is None:
        quality_report = assess_data_quality(df, save_path=REPORTS_DIR / "_tmp_quality.json")

    # Собираем статистики по текущему состоянию (после очистки + FE)
    current_stats = _compute_current_stats(df)

    # Рекомендации
    recommendations = _generate_recommendations(quality_report, cleaning_info, current_stats)

    # Рендерим HTML
    html = _render_quality_html(quality_report, cleaning_info, current_stats, recommendations)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)

    LOGGER.info(f"Data quality report saved: {output_path}")
    return output_path


def _compute_current_stats(df: pd.DataFrame) -> dict:
    """Статистики текущего состояния DataFrame (после очистки и FE)."""
    stats = {
        "shape": {"n_rows": len(df), "n_cols": len(df.columns)},
        "columns": list(df.columns),
    }

    # Пропуски
    missing = df.isnull().sum()
    stats["missing"] = {
        col: {"count": int(v), "percent": round(float(v / len(df) * 100), 2)}
        for col, v in missing.items() if v > 0
    }
    stats["total_missing"] = int(missing.sum())

    # Типы колонок
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    stats["numeric_features"] = [c for c in numeric_cols if c != "label"]
    stats["categorical_features"] = cat_cols

    # Label баланс
    if "label" in df.columns:
        vc = df["label"].value_counts()
        stats["label_counts"] = {str(k): int(v) for k, v in vc.items()}
        stats["label_balance_ratio"] = round(float(vc.min() / vc.max()), 4) if vc.max() > 0 else 0

    # NaN-тяжёлые колонки
    nan_heavy = {col: round(float(v / len(df) * 100), 2) for col, v in missing.items() if v / len(df) > 0.05}
    stats["nan_heavy_columns"] = nan_heavy

    # Числовые колонки с большим количеством выбросов (по IQR)
    outlier_info = {}
    for col in stats["numeric_features"]:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        n_outliers = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
        if n_outliers > 0:
            outlier_info[col] = {"count": n_outliers, "percent": round(n_outliers / len(s) * 100, 2)}
    stats["outliers_iqr"] = outlier_info

    return stats


def _generate_recommendations(quality_report: dict, cleaning_info: dict, current_stats: dict) -> list:
    # Генерирует список рекомендаций по улучшению качества данных
    recs = []

    # Пропуски
    if current_stats["total_missing"] > 0:
        for col, info in current_stats["missing"].items():
            if info["percent"] > 5:
                recs.append(f" Column '{col}' has {info['percent']}% missing values — consider imputation or removal.")
            elif info["percent"] > 0:
                recs.append(f"ℹ Column '{col}' has {info['count']} missing values ({info['percent']}%).")
    else:
        recs.append(" No missing values in the dataset after cleaning.")

    # Баланс классов
    if "label_balance_ratio" in current_stats:
        ratio = current_stats["label_balance_ratio"]
        if ratio < 0.5:
            recs.append(f" Class imbalance detected (minority/majority ratio = {ratio:.2f}). "
                        f"Consider oversampling, undersampling, or class weights.")
        else:
            recs.append(f" Classes are reasonably balanced (ratio = {ratio:.2f}).")

    # Выбросы
    if current_stats["outliers_iqr"]:
        heavy_outliers = {k: v for k, v in current_stats["outliers_iqr"].items() if v["percent"] > 5}
        if heavy_outliers:
            cols = ", ".join(heavy_outliers.keys())
            recs.append(f"Significant outliers (>5% by IQR) in: {cols}. Review before modeling.")
    else:
        recs.append(" No significant outliers detected by IQR method.")

    # NaN-тяжёлые
    if current_stats["nan_heavy_columns"]:
        cols = ", ".join(current_stats["nan_heavy_columns"].keys())
        recs.append(f" Columns with >5% NaN: {cols}.")

    # Информация из отчёта очистки
    if cleaning_info:
        removed_pct = cleaning_info.get("removal_percent", 0)
        if removed_pct > 10:
            recs.append(f" Cleaning removed {removed_pct}% of data — verify thresholds are not too aggressive.")
        elif removed_pct > 0:
            recs.append(f"  Cleaning removed {removed_pct}% of data — within acceptable range.")

    return recs


def _render_quality_html(
    quality_report: dict,
    cleaning_info: dict,
    current_stats: dict,
    recommendations: list,
) -> str:
    # Рендерит сводный HTML-отчёт

    # Раздел с пропусками
    if quality_report.get("missing"):
        missing_rows = ""
        for col, info in quality_report["missing"].items():
            color = "#e74c3c" if info["percent"] > 5 else ("#f39c12" if info["percent"] > 0 else "#2ecc71")
            missing_rows += f"""
            <tr>
                <td>{col}</td>
                <td>{info['count']}</td>
                <td style="color: {color}; font-weight: bold;">{info['percent']:.2f}%</td>
            </tr>"""
    else:
        missing_rows = "<tr><td colspan='3'>No data</td></tr>"

    # Отчёт по очистке
    cleaning_rows = ""
    if cleaning_info.get("steps"):
        for step in cleaning_info["steps"]:
            icon = "Bad" if step["removed"] > 0 else "Good"
            cleaning_rows += f"""
            <tr>
                <td>{icon} {step['step']}</td>
                <td>{step['removed']}</td>
                <td>{step['remaining']}</td>
            </tr>"""

    # Таблица с выбросами
    outlier_rows = ""
    if current_stats.get("outliers_iqr"):
        for col, info in current_stats["outliers_iqr"].items():
            outlier_rows += f"""
            <tr>
                <td>{col}</td>
                <td>{info['count']}</td>
                <td>{info['percent']:.2f}%</td>
            </tr>"""
    else:
        outlier_rows = "<tr><td colspan='3'>No significant outliers</td></tr>"

    # Рекомендации
    rec_items = "\n".join(f"<li>{r}</li>" for r in recommendations)

    # Распределение классов
    label_html = ""
    if "label_distribution" in quality_report:
        label_items = "".join(
            f"<li><strong>{k}</strong>: {v}</li>"
            for k, v in quality_report["label_distribution"].items()
        )
        label_html = f"""
        <h2> Target Variable</h2>
        <ul>{label_items}</ul>
        <p>Balance (mean label): <strong>{quality_report.get('label_balance', 'N/A')}</strong></p>
        """

    # Распределение по subject
    subject_html = ""
    if "subject_distribution" in quality_report:
        subject_items = "".join(
            f"<li><strong>{k}</strong>: {v}</li>"
            for k, v in quality_report["subject_distribution"].items()
        )
        subject_html = f"""
        <h2> Subject Distribution</h2>
        <ul>{subject_items}</ul>
        """

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Data Quality Report</title>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 40px; background: #f8f9fa; color: #333; }}
        h1 {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }}
        h2 {{ color: #34495e; margin-top: 30px; }}
        table {{ border-collapse: collapse; width: 100%; margin: 15px 0; background: white; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
        th {{ background: #3498db; color: white; padding: 10px 15px; text-align: left; }}
        td {{ padding: 8px 15px; border-bottom: 1px solid #ecf0f1; }}
        tr:hover {{ background: #f5f6fa; }}
        .summary-box {{ background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); margin: 15px 0; }}
        .metric {{ display: inline-block; margin: 10px 20px; text-align: center; }}
        .metric .value {{ font-size: 28px; font-weight: bold; color: #2c3e50; }}
        .metric .label {{ font-size: 13px; color: #7f8c8d; }}
        .rec-list {{ background: white; padding: 15px 25px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
        .rec-list li {{ margin: 8px 0; line-height: 1.5; }}
        .timestamp {{ color: #95a5a6; font-size: 12px; margin-top: 40px; }}
    </style>
</head>
<body>
    <h1> Data Quality Report — Fake News Dataset</h1>

    <div class="summary-box">
        <div class="metric"><div class="value">{quality_report['shape']['n_rows']}</div><div class="label">Rows (raw)</div></div>
        <div class="metric"><div class="value">{current_stats['shape']['n_rows']}</div><div class="label">Rows (cleaned)</div></div>
        <div class="metric"><div class="value">{current_stats['shape']['n_cols']}</div><div class="label">Columns</div></div>
        <div class="metric"><div class="value">{len(current_stats.get('numeric_features', []))}</div><div class="label">Numeric features</div></div>
        <div class="metric"><div class="value">{current_stats['total_missing']}</div><div class="label">Missing cells</div></div>
    </div>

    <h2> Missing Values (raw data)</h2>
    <table>
        <tr><th>Column</th><th>Count</th><th>Percent</th></tr>
        {missing_rows}
    </table>

    {label_html}
    {subject_html}

    <h2> Cleaning Steps</h2>
    {"<p>Before: <strong>" + str(cleaning_info.get('rows_before', 'N/A')) + "</strong> → After: <strong>" + str(cleaning_info.get('rows_after', 'N/A')) + "</strong> (removed " + str(cleaning_info.get('total_removed', 'N/A')) + ", " + str(cleaning_info.get('removal_percent', 'N/A')) + "%)</p>" if cleaning_info else "<p>No cleaning info available.</p>"}
    <table>
        <tr><th>Step</th><th>Removed</th><th>Remaining</th></tr>
        {cleaning_rows}
    </table>

    <h2> Outliers (IQR method, after cleaning)</h2>
    <table>
        <tr><th>Feature</th><th>Count</th><th>Percent</th></tr>
        {outlier_rows}
    </table>

    <h2> Recommendations</h2>
    <div class="rec-list">
        <ul>{rec_items}</ul>
    </div>

    <p class="timestamp">Report generated at: {datetime.now().isoformat()}</p>
</body>
</html>"""

    return html

In [102]:
generate_quality_report(df, quality_report=quality_report)

/tmp/ipykernel_16991/4251568379.py:57: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
2026-03-22 02:29:38,261 | INFO | Data quality report saved: reports/data_quality_report.html


PosixPath('reports/data_quality_report.html')

In [112]:
# Числовые колонки, создаваемые в add_features (без label)
NUMERIC_FEATURES = [
    "text_len_chars", "text_len_words", "text_n_sentences",
    "text_avg_word_len", "text_avg_sentence_len",
    "title_len_chars", "title_len_words", "title_to_text_ratio",
    "upper_char_count", "upper_ratio", "caps_word_count", "caps_word_ratio",
    "exclamation_count", "question_count", "ellipsis_count", "quote_count",
    "punct_count", "punct_ratio",
    "number_count", "type_token_ratio", "hapax_ratio",
]

BINARY_FEATURES = [
    "has_trump", "has_aggression", "has_url", "has_email", "has_number",
    "has_breaking", "has_exclusive", "has_shocking", "is_weekend",
]

CATEGORICAL_FEATURES = ["subject"]

TEMPORAL_FEATURES = ["day_of_week", "month", "year"]


def handle_missing_values(
    df: pd.DataFrame,
    numeric_strategy: str = "median",
    add_nan_flags: bool = True,
) -> pd.DataFrame:
    """
    3a.i — Обработка пропусков.
    Стратегии для числовых: 'median', 'mean', 'zero'.
    Для категориальных: заполнение значением 'unknown'.
    Для бинарных: заполнение 0 (отсутствие признака).
    Опционально добавляет флаг *_was_nan для колонок, где были пропуски.

    Аргументы:
        df: DataFrame с признаками (после add_features)
        numeric_strategy: стратегия заполнения числовых ('median', 'mean', 'zero')
        add_nan_flags: добавлять ли бинарные флаги *_was_nan
    """
    df = df.copy()
    fill_log = []

    # Числовые колонки (включая временные)
    num_cols = [c for c in NUMERIC_FEATURES + TEMPORAL_FEATURES if c in df.columns]
    for col in num_cols:
        n_miss = df[col].isna().sum()
        if n_miss == 0:
            continue

        if add_nan_flags:
            df[f"{col}_was_nan"] = df[col].isna().astype(int)

        if numeric_strategy == "median":
            fill_val = df[col].median()
        elif numeric_strategy == "mean":
            fill_val = df[col].mean()
        elif numeric_strategy == "zero":
            fill_val = 0
        else:
            raise ValueError(f"Unknown numeric_strategy: {numeric_strategy}")

        df[col] = df[col].fillna(fill_val)
        fill_log.append({"column": col, "missing": n_miss, "strategy": numeric_strategy, "fill_value": round(float(fill_val), 4)})

    # Бинарные колонки 
    bin_cols = [c for c in BINARY_FEATURES if c in df.columns]
    for col in bin_cols:
        n_miss = df[col].isna().sum()
        if n_miss == 0:
            continue
        df[col] = df[col].fillna(0).astype(int)
        fill_log.append({"column": col, "missing": n_miss, "strategy": "zero", "fill_value": 0})

    # Категориальные колонки
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]
    for col in cat_cols:
        n_miss = df[col].isna().sum()
        if n_miss == 0:
            continue

        if add_nan_flags:
            df[f"{col}_was_nan"] = df[col].isna().astype(int)

        df[col] = df[col].fillna("unknown")
        fill_log.append({"column": col, "missing": n_miss, "strategy": "fill_unknown", "fill_value": "unknown"})

    # title / date (вспомогательные, не для модели)
    if "title" in df.columns:
        df["title"] = df["title"].fillna("")
    if "date" in df.columns:
        df["date"] = df["date"].fillna("unknown")

    # Лог
    if fill_log:
        LOGGER.info(f"Missing value handling: filled {len(fill_log)} columns")
        for entry in fill_log:
            LOGGER.info(f"  {entry['column']}: {entry['missing']} NaN → {entry['strategy']} ({entry['fill_value']})")
    else:
        LOGGER.info("Missing value handling: no missing values found")

    remaining = df.isnull().sum().sum()
    LOGGER.info(f"Remaining NaN after imputation: {remaining}")

    return df


def encode_categorical(
    df: pd.DataFrame,
    method: str = "onehot",
) -> pd.DataFrame:
    """
    3a.ii — Обработка категориальных переменных.
    Методы:
      - 'onehot': One-Hot Encoding (pd.get_dummies), drop_first=True
      - 'label': Label Encoding (числовые коды)
      - 'frequency': Frequency Encoding (доля каждой категории)
    Аргументы:
        df: DataFrame
        method: метод кодирования
    """
    df = df.copy()
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]

    if not cat_cols:
        LOGGER.info("Categorical encoding: no categorical columns found")
        return df

    LOGGER.info(f"Categorical encoding: method='{method}', columns={cat_cols}")

    if method == "onehot":
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    elif method == "label":
        for col in cat_cols:
            categories = sorted(df[col].unique())
            cat_map = {cat: i for i, cat in enumerate(categories)}
            df[col] = df[col].map(cat_map)
            LOGGER.info(f"  {col}: {cat_map}")

    elif method == "frequency":
        for col in cat_cols:
            freq = df[col].value_counts(normalize=True)
            df[f"{col}_freq"] = df[col].map(freq).round(4)
            df = df.drop(columns=[col])
            LOGGER.info(f"  {col}: replaced with {col}_freq")

    else:
        raise ValueError(f"Unknown encoding method: {method}")

    return df


def scale_numeric(
    df: pd.DataFrame,
    method: str = "standard",
    exclude_binary: bool = True,
) -> pd.DataFrame:
    """
    3a.iii — Обработка числовых переменных (масштабирование).
    Методы:
      - 'standard': StandardScaler (z-score)
      - 'minmax': MinMaxScaler [0, 1]
      - 'robust': RobustScaler (устойчив к выбросам)
      - 'none': без масштабирования
    Аргументы:
        df: DataFrame
        method: метод масштабирования
        exclude_binary: не масштабировать бинарные признаки
    """
    df = df.copy()

    if method == "none":
        LOGGER.info("Numeric scaling: skipped (method='none')")
        return df

    num_cols = [c for c in NUMERIC_FEATURES + TEMPORAL_FEATURES if c in df.columns]

    if exclude_binary:
        # Не масштабируем колонки, где только 0 и 1
        bin_in_numeric = [c for c in num_cols if df[c].dropna().isin([0, 1]).all()]
        num_cols = [c for c in num_cols if c not in bin_in_numeric]

    if not num_cols:
        LOGGER.info("Numeric scaling: no columns to scale")
        return df

    LOGGER.info(f"Numeric scaling: method='{method}', {len(num_cols)} columns")

    if method == "standard":
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
    elif method == "minmax":
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    elif method == "robust":
        from sklearn.preprocessing import RobustScaler
        scaler = RobustScaler()
    else:
        raise ValueError(f"Unknown scaling method: {method}")

    df[num_cols] = scaler.fit_transform(df[num_cols])
    LOGGER.info(f"  Scaled columns: {num_cols}")

    return df


def prepare_data(
    df: pd.DataFrame,
    numeric_impute: str = "median",
    categorical_encoding: str = "onehot",
    scaling: str = "standard",
    add_nan_flags: bool = True,
    drop_raw_text: bool = False,
) -> pd.DataFrame:
    """
      1. Обработка пропусков (3a.i)
      2. Кодирование категориальных переменных (3a.ii)
      3. Масштабирование числовых переменных (3a.iii)
    Аргументы:
        df: DataFrame после add_features()
        numeric_impute: стратегия заполнения пропусков ('median', 'mean', 'zero')
        categorical_encoding: метод кодирования ('onehot', 'label', 'frequency')
        scaling: метод масштабирования ('standard', 'minmax', 'robust', 'none')
        add_nan_flags: добавлять ли флаги *_was_nan
        drop_raw_text: удалить ли сырые text/title/date колонки после подготовки (по умолчанию True, так как они не нужны для табличных моделей)
    """
    LOGGER.info("=" * 50)
    LOGGER.info("BLOCK 3: Data Preparation Pipeline")
    LOGGER.info(f"  impute={numeric_impute}, encoding={categorical_encoding}, "
                f"scaling={scaling}, nan_flags={add_nan_flags}")
    LOGGER.info("=" * 50)

    # 3a.i — Пропуски
    df = handle_missing_values(df, numeric_strategy=numeric_impute, add_nan_flags=add_nan_flags)

    # 3a.ii — Категориальные
    df = encode_categorical(df, method=categorical_encoding)

    # 3a.iii — Числовые
    df = scale_numeric(df, method=scaling)

    # Убираем сырые текстовые колонки (не нужны для табличных моделей)
    if drop_raw_text:
        drop_cols = [c for c in ["text", "title", "date", "source"] if c in df.columns]
        df = df.drop(columns=drop_cols)
        LOGGER.info(f"Dropped raw columns: {drop_cols}")

    LOGGER.info(f"Final shape: {df.shape}")
    LOGGER.info(f"Final columns: {list(df.columns)}")

    _print_preparation_summary(df)
    return df


def _print_preparation_summary(df: pd.DataFrame):
    # Выводит сводку после подготовки данных
    print(f"\n{'=' * 60}")
    print("  DATA PREPARATION SUMMARY")
    print(f"{'=' * 60}")
    print(f"\n  Shape: {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"  Remaining NaN: {df.isnull().sum().sum()}")

    dtypes = df.dtypes.value_counts()
    print(f"  Dtypes: {dict(dtypes)}")

    if "label" in df.columns:
        feature_cols = [c for c in df.columns if c != "label"]
        print(f"  Features: {len(feature_cols)}")
        print(f"  Target: label ({df['label'].value_counts().to_dict()})")

    print(f"\n{'=' * 60}")

In [113]:
df_prepared = prepare_data(df)

2026-03-22 02:57:48,363 | INFO | ==================================================
2026-03-22 02:57:48,364 | INFO | BLOCK 3: Data Preparation Pipeline
2026-03-22 02:57:48,364 | INFO |   impute=median, encoding=onehot, scaling=standard, nan_flags=True
2026-03-22 02:57:48,365 | INFO | ==================================================
2026-03-22 02:57:48,389 | INFO | Missing value handling: filled 1 columns
2026-03-22 02:57:48,390 | INFO |   text_avg_sentence_len: 83 NaN → median (19.14)
2026-03-22 02:57:48,401 | INFO | Remaining NaN after imputation: 0
2026-03-22 02:57:48,409 | INFO | Categorical encoding: method='onehot', columns=['subject']
2026-03-22 02:57:48,466 | INFO | Numeric scaling: method='standard', 24 columns
2026-03-22 02:57:48,482 | INFO |   Scaled columns: ['text_len_chars', 'text_len_words', 'text_n_sentences', 'text_avg_word_len', 'text_avg_sentence_len', 'title_len_chars', 'title_len_words', 'title_to_text_ratio', 'upper_char_count', 'upper_ratio', 'caps_word_count', 


  DATA PREPARATION SUMMARY

  Shape: 30794 rows × 45 cols
  Remaining NaN: 0
  Dtypes: {dtype('float64'): np.int64(24), dtype('int64'): np.int64(17), <StringDtype(storage='python', na_value=nan)>: np.int64(4)}
  Features: 44
  Target: label ({0: 16952, 1: 13842})



In [130]:
df_prepared.head()

,title,text,date,label,source,text_len_chars,text_len_words,text_n_sentences,text_avg_word_len,text_avg_sentence_len,...,month,year,is_weekend,text_avg_sentence_len_was_nan,subject_News,subject_US_News,subject_left-news,subject_politics,subject_politicsNews,subject_worldnews
0,WATCH SHOCKING DISPLAY of Muslim Intimidation ...,This is in-your-face taunting of our president...,2017-06-04,1,train1,-0.322955,-0.336367,0.233647,0.067239,-1.369862,...,-0.369368,0.765979,1,0,0,0,0,1,0,0
1,Two Florida Lawmakers Just Voted Against Hurri...,Two Florida Republican lawmakers voted against...,2017-09-09,1,train1,0.352327,0.365503,0.745275,-0.066229,-0.832742,...,0.483301,0.765979,1,0,1,0,0,0,0,0
2,BOOM! HARRIS FAULKNER Blows Up The Russia Coll...,Fox News Harris Faulkner BLOWS UP the WHOLE R...,2017-05-29,1,train1,-0.316097,-0.300624,-0.448524,-0.199698,0.433445,...,-0.653591,0.765979,0,0,0,0,0,1,0,0
3,House of Representatives to try again to seek ...,WASHINGTON (Reuters) - The U.S. House of Repre...,2016-01-13,0,train1,0.095931,0.115299,0.233647,-0.140378,-0.470734,...,-1.790483,-0.975571,0,0,0,0,0,0,1,0
4,German citizen on trial in Turkey on political...,FRANKFURT (Reuters) - A 49-year-old woman has ...,2017-09-07,0,train1,-0.434798,-0.424101,-0.448524,-0.184868,-0.024538,...,0.483301,0.765979,0,0,0,0,0,0,0,1


In [131]:
import re
import string


def _lowercase(text: str) -> str:
    return text.lower()


def _remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans("", "", string.punctuation))


def _remove_numbers(text: str) -> str:
    return re.sub(r"\d+", "", text)


def _remove_urls(text: str) -> str:
    return re.sub(r"https?://\S+|www\.\S+", "", text)


def _remove_extra_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def _remove_stopwords(text: str) -> str:
    # Удаление стоп-слов (английский, без внешних зависимостей)
    # Минимальный набор английских стоп-слов
    stopwords = {
        "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
        "of", "with", "by", "from", "is", "was", "are", "were", "be", "been",
        "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "could", "should", "may", "might", "shall", "can", "not", "no", "nor",
        "so", "if", "then", "than", "that", "this", "these", "those", "it",
        "its", "i", "me", "my", "we", "our", "you", "your", "he", "him",
        "his", "she", "her", "they", "them", "their", "what", "which", "who",
        "whom", "when", "where", "how", "all", "each", "every", "both",
        "few", "more", "most", "other", "some", "such", "only", "own",
        "same", "just", "about", "above", "after", "again", "also", "as",
        "because", "before", "between", "during", "into", "over", "under",
        "up", "down", "out", "off", "very", "too", "here", "there",
    }
    words = text.split()
    return " ".join(w for w in words if w.lower() not in stopwords)


def _stem_text(text: str) -> str:
    # Простой суффиксный стемминг (Porter-lite, без NLTK)
    def _simple_stem(word: str) -> str:
        for suffix in ["ation", "tion", "sion", "ment", "ness", "ence", "ance",
                        "ible", "able", "ful", "less", "ous", "ive", "ing",
                        "ies", "ed", "er", "ly", "es", "s"]:
            if word.endswith(suffix) and len(word) - len(suffix) >= 3:
                return word[:-len(suffix)]
        return word
    return " ".join(_simple_stem(w) for w in text.split())


# Определение вариантов предобработки текста
TEXT_PREPROCESSING_VARIANTS = {
    "raw": {
        "description": "Без предобработки — исходный текст как есть",
        "steps": [],
    },
    "basic_clean": {
        "description": "Базовая очистка: lowercase + удаление URL + лишние пробелы",
        "steps": [_lowercase, _remove_urls, _remove_extra_whitespace],
    },
    "normalized": {
        "description": "Нормализация: lowercase + удаление URL, пунктуации, чисел + пробелы",
        "steps": [_lowercase, _remove_urls, _remove_punctuation, _remove_numbers, _remove_extra_whitespace],
    },
    "no_stopwords": {
        "description": "Нормализация + удаление стоп-слов",
        "steps": [_lowercase, _remove_urls, _remove_punctuation, _remove_numbers,
                  _remove_extra_whitespace, _remove_stopwords],
    },
    "stemmed": {
        "description": "Нормализация + удаление стоп-слов + стемминг",
        "steps": [_lowercase, _remove_urls, _remove_punctuation, _remove_numbers,
                  _remove_extra_whitespace, _remove_stopwords, _stem_text],
    },
}


def _apply_text_pipeline(text: str, steps: list) -> str:
    # Последовательно применяет список функций к тексту
    for fn in steps:
        text = fn(text)
    return text


def create_text_preprocessing_variants(
    df: pd.DataFrame,
    variants: dict = None,
    text_col: str = "text",
    save_dir: Path = None,
) -> dict:
    """
    Создаёт несколько вариантов предобработки текста.
    Каждый вариант — это отдельный DataFrame с обработанной колонкой text,
    сохранённый в CSV для дальнейшего перебора при обучении модели.
    Аргументы:
        df: DataFrame с колонкой text (после clean_data / add_features)
        variants: словарь {name: {description, steps}}
        text_col: название колонки с текстом
        save_dir: директория для сохранения CSV
    """
    if variants is None:
        variants = TEXT_PREPROCESSING_VARIANTS
    if save_dir is None:
        save_dir = Path("data/prepared")
    save_dir.mkdir(parents=True, exist_ok=True)

    if text_col not in df.columns:
        LOGGER.warning(f"Column '{text_col}' not found — skipping text variant creation")
        return {}

    results = {}

    LOGGER.info(f"Creating {len(variants)} text preprocessing variants...")

    for name, config in variants.items():
        LOGGER.info(f"  Variant '{name}': {config['description']}")

        df_variant = df.copy()
        steps = config["steps"]

        if steps:
            df_variant[text_col] = df_variant[text_col].fillna("").apply(
                lambda t: _apply_text_pipeline(t, steps)
            )
        # Иначе (raw) — оставляем как есть

        results[name] = df_variant

        csv_path = save_dir / f"text_{name}.csv"
        df_variant.to_csv(csv_path, index=False)
        LOGGER.info(f"    Saved: {csv_path}")

    _print_text_variants_summary(df, results, variants, text_col)

    return results


def _print_text_variants_summary(
    df_original: pd.DataFrame,
    results: dict,
    variants: dict,
    text_col: str,
):
    # Сводка по вариантам текстовой предобработки с примерами
    print(f"\n{'=' * 70}")
    print("  TEXT PREPROCESSING VARIANTS")
    print(f"{'=' * 70}")

    # Берём пример текста для демонстрации
    sample_idx = df_original[text_col].fillna("").str.len().idxmax()
    sample_original = df_original.loc[sample_idx, text_col]
    # Обрезаем для вывода
    sample_short = sample_original[:150] + "..." if len(str(sample_original)) > 150 else str(sample_original)
    print(f"\n  Sample (original, first 150 chars):\n    \"{sample_short}\"")

    for name, df_v in results.items():
        config = variants[name]
        text_lens = df_v[text_col].fillna("").str.len()
        word_counts = df_v[text_col].fillna("").str.split().str.len()

        sample_processed = str(df_v.loc[sample_idx, text_col])[:150]
        if len(str(df_v.loc[sample_idx, text_col])) > 150:
            sample_processed += "..."

        print(f"\n  {name}: {config['description']}")
        print(f"    Steps: {len(config['steps'])}")
        print(f"    Avg chars: {text_lens.mean():.0f}, Avg words: {word_counts.mean():.0f}")
        print(f"    Sample: \"{sample_processed}\"")

    print(f"\n{'=' * 70}")

In [132]:
text_variants = create_text_preprocessing_variants(df)

2026-03-22 03:15:36,901 | INFO | Creating 5 text preprocessing variants...
2026-03-22 03:15:36,903 | INFO |   Variant 'raw': Без предобработки — исходный текст как есть
2026-03-22 03:15:38,368 | INFO |     Saved: data/prepared/text_raw.csv
2026-03-22 03:15:38,369 | INFO |   Variant 'basic_clean': Базовая очистка: lowercase + удаление URL + лишние пробелы
2026-03-22 03:15:43,480 | INFO |     Saved: data/prepared/text_basic_clean.csv
2026-03-22 03:15:43,481 | INFO |   Variant 'normalized': Нормализация: lowercase + удаление URL, пунктуации, чисел + пробелы
2026-03-22 03:15:52,042 | INFO |     Saved: data/prepared/text_normalized.csv
2026-03-22 03:15:52,043 | INFO |   Variant 'no_stopwords': Нормализация + удаление стоп-слов
2026-03-22 03:16:02,575 | INFO |     Saved: data/prepared/text_no_stopwords.csv
2026-03-22 03:16:02,576 | INFO |   Variant 'stemmed': Нормализация + удаление стоп-слов + стемминг
2026-03-22 03:16:18,853 | INFO |     Saved: data/prepared/text_stemmed.csv



  TEXT PREPROCESSING VARIANTS

  Sample (original, first 150 chars):
    "We just discovered another reason NOT to support the NFL The man who is the most anti-American person we know is now connected to The National Footbal..."

  raw: Без предобработки — исходный текст как есть
    Steps: 0
    Avg chars: 2463, Avg words: 405
    Sample: "We just discovered another reason NOT to support the NFL The man who is the most anti-American person we know is now connected to The National Footbal..."

  basic_clean: Базовая очистка: lowercase + удаление URL + лишние пробелы
    Steps: 3
    Avg chars: 2451, Avg words: 404
    Sample: "we just discovered another reason not to support the nfl the man who is the most anti-american person we know is now connected to the national footbal..."

  normalized: Нормализация: lowercase + удаление URL, пунктуации, чисел + пробелы
    Steps: 5
    Avg chars: 2375, Avg words: 399
    Sample: "we just discovered another reason not to support the nfl the man

In [133]:
text_variants

{'raw':                                                    title  \
 0      WATCH SHOCKING DISPLAY of Muslim Intimidation ...   
 1      Two Florida Lawmakers Just Voted Against Hurri...   
 2      BOOM! HARRIS FAULKNER Blows Up The Russia Coll...   
 3      House of Representatives to try again to seek ...   
 4      German citizen on trial in Turkey on political...   
 ...                                                  ...   
 30789       A picture and its story: tear gas in Nairobi   
 30790  WATCH: INTOLERANT GAY COFFEE SHOP Owner Scream...   
 30791  JUST IN: Anti-Putin Banker Claims Firm Tied To...   
 30792  Lebanese president calls Hariri's situation in...   
 30793  Trump has 'warm rapport' with Philippines' Dut...   
 
                                                     text       subject  \
 0      This is in-your-face taunting of our president...      politics   
 1      Two Florida Republican lawmakers voted against...          News   
 2      Fox News  Harris Faulkner 

In [134]:
def run_analysis_and_preparation(train1_path: str, train2_path: str):
    # 0. Загрузка и объединение
    df = load_and_merge(train1_path, train2_path)
    
    # === Анализ данных ===
    
    # a.i — Оценка качества данных (data quality)
    quality_report = assess_data_quality(df)
    
    # a.ii — Ассоциативные правила (Apriori / FP-tree)
    rules = mine_association_rules(df)
    
    # a.iii — Базовая очистка по порогам качества
    df = clean_data(df)
    
    # b.i — Автоматический EDA
    run_automatic_eda(df)
    
    # b.ii — Feature Engineering
    df = add_features(df)
    _print_feature_summary(df)
    
    # b.iii — Отчёт о качестве данных
    generate_quality_report(df, quality_report=quality_report)
    
    # b.iv — Data Drift (train1 vs train2)
    # TODO
    
    # === Подготовка данных ===
    
    # a.i — Обработка пропусков + a.ii — Обработка категориальных переменных + a.iii — Обработка числовых переменных
    df_prepared = prepare_data(df)

    # b.i — Несколько вариантов предобработки
    
    text_variants = create_text_preprocessing_variants(df)
    
    return df, df_prepared

In [136]:
if __name__ == "__main__":
    df = run_analysis_and_preparation(
        "data/processed/train1.csv",
        "data/processed/train2.csv",
    )

2026-03-22 03:19:16,541 | INFO | train1: (23092, 6), train2: (7815, 6), combined: (30907, 6)
2026-03-22 03:19:18,013 | INFO | Data quality report saved: reports/data_quality.json



  DATA QUALITY REPORT

 Shape: 30907 rows × 6 cols

 Missing values:
   text: 1 (0.00%)

 Duplicates:
   Full row: 0
   By text:  0

 Label distribution:
   0: 16952
   1: 13955
   Balance (mean label): 0.4515

 Text length (chars): mean=2455.95, median=2230.0, min=0, max=51793
   Text length (words): mean=403.32, median=369.0, min=0, max=8135
   Empty texts: 1, short (<50 chars): 112

 Subject distribution:
   politicsNews: 8986
   worldnews: 7966
   News: 7230
   politics: 5138
   US_News: 629
   left-news: 545
   Government News: 413

 Date range: 2015-03-31 — 2018-02-19



2026-03-22 03:19:20,434 | INFO | Created 18 binary features: ['is_fake', 'is_real', 'has_trump', 'has_aggression', 'is_short_text', 'is_long_text', 'is_medium_text', 'has_missing_date', 'has_empty_title', 'subject_politics', 'subject_news', 'subject_politicsnews', 'subject_worldnews', 'subject_left_news', 'subject_government_news', 'subject_us_news', 'source_train1', 'source_train2']
2026-03-22 03:19:20,440 | INFO | Running FP-Growth: min_support=0.05
2026-03-22 03:19:20,559 | INFO | Found 226 frequent itemsets
2026-03-22 03:19:20,566 | INFO | Generated 736 rules (confidence >= 0.5)
2026-03-22 03:19:20,568 | INFO | After lift >= 1.2: 566 rules
2026-03-22 03:19:20,573 | INFO | Selected 5 interesting rules
2026-03-22 03:19:20,577 | INFO | Association rules report saved: reports/association_rules.json
2026-03-22 03:19:20,581 | INFO | Starting data cleaning: 30907 rows
2026-03-22 03:19:20,598 | INFO |   [drop_missing_text] removed 1 rows → 30906 remaining
2026-03-22 03:19:20,608 | INFO |  


  SELECTED RULES (for data quality / feature enrichment)

  Rule 1:
    is_fake, is_short_text : source_train1, subject_politics
    support=0.0580  confidence=0.7331  lift=4.9702

  Rule 2:
    subject_news : is_fake
    support=0.2339  confidence=1.0000  lift=2.2148

  Rule 3:
    has_aggression, is_long_text, is_real, subject_politicsnews : has_trump
    support=0.0671  confidence=0.8452  lift=1.9825

  Rule 4:
    is_long_text, subject_politicsnews : is_real
    support=0.0875  confidence=1.0000  lift=1.8232

  Rule 5:
    is_long_text, is_real, subject_worldnews : has_aggression
    support=0.0562  confidence=0.9554  lift=1.3038


  TOP-15 RULES BY LIFT

  Rule 1:
    is_long_text, subject_politicsnews : has_aggression, has_trump, is_real, source_train1
    support=0.0590  confidence=0.6736  lift=5.2346

  Rule 2:
    is_fake, is_short_text : source_train1, subject_politics
    support=0.0580  confidence=0.7331  lift=4.9702

  Rule 3:
    is_long_text, source_train1, subject_poli

2026-03-22 03:19:30,786 | INFO | Sampling 5000 rows from 30794 for EDA
2026-03-22 03:19:30,791 | INFO | Generating sweetviz report (fake vs real comparison)...
Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)
2026-03-22 03:19:37,580 | INFO | sweetviz report saved: reports/eda_sweetviz.html
2026-03-22 03:19:37,583 | INFO | Generating matplotlib EDA report...
/tmp/ipykernel_16991/1801632230.py:164: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  ct.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white")
/tmp/ipykernel_16991/1801632230.py:164: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies u

...


  FEATURE ENGINEERING SUMMARY

  Total features: 33
  Binary features (9): ['has_trump', 'has_aggression', 'has_url', 'has_email', 'has_number', 'has_breaking', 'has_exclusive', 'has_shocking', 'is_weekend']
  Continuous features (24): ['text_len_chars', 'text_len_words', 'text_n_sentences', 'text_avg_word_len', 'text_avg_sentence_len', 'title_len_chars', 'title_len_words', 'title_to_text_ratio', 'upper_char_count', 'upper_ratio', 'caps_word_count', 'caps_word_ratio', 'exclamation_count', 'question_count', 'ellipsis_count', 'quote_count', 'punct_count', 'punct_ratio', 'number_count', 'type_token_ratio', 'hapax_ratio', 'day_of_week', 'month', 'year']

  Binary feature rates:
    has_trump: 42.77%
    has_aggression: 73.53%
    has_url: 6.37%
    has_email: 0.09%
    has_number: 81.42%
    has_breaking: 2.18%
    has_exclusive: 0.59%
    has_shocking: 2.57%
    is_weekend: 18.45%



2026-03-22 03:20:07,238 | INFO | Numeric scaling: method='standard', 24 columns
2026-03-22 03:20:07,262 | INFO |   Scaled columns: ['text_len_chars', 'text_len_words', 'text_n_sentences', 'text_avg_word_len', 'text_avg_sentence_len', 'title_len_chars', 'title_len_words', 'title_to_text_ratio', 'upper_char_count', 'upper_ratio', 'caps_word_count', 'caps_word_ratio', 'exclamation_count', 'question_count', 'ellipsis_count', 'quote_count', 'punct_count', 'punct_ratio', 'number_count', 'type_token_ratio', 'hapax_ratio', 'day_of_week', 'month', 'year']
2026-03-22 03:20:07,264 | INFO | Final shape: (30794, 45)
2026-03-22 03:20:07,265 | INFO | Final columns: ['title', 'text', 'date', 'label', 'source', 'text_len_chars', 'text_len_words', 'text_n_sentences', 'text_avg_word_len', 'text_avg_sentence_len', 'title_len_chars', 'title_len_words', 'title_to_text_ratio', 'upper_char_count', 'upper_ratio', 'caps_word_count', 'caps_word_ratio', 'exclamation_count', 'question_count', 'ellipsis_count', 'qu


  DATA PREPARATION SUMMARY

  Shape: 30794 rows × 45 cols
  Remaining NaN: 0
  Dtypes: {dtype('float64'): np.int64(24), dtype('int64'): np.int64(17), <StringDtype(storage='python', na_value=nan)>: np.int64(4)}
  Features: 44
  Target: label ({0: 16952, 1: 13842})



2026-03-22 03:20:08,789 | INFO |     Saved: data/prepared/text_raw.csv
2026-03-22 03:20:08,790 | INFO |   Variant 'basic_clean': Базовая очистка: lowercase + удаление URL + лишние пробелы
2026-03-22 03:20:13,916 | INFO |     Saved: data/prepared/text_basic_clean.csv
2026-03-22 03:20:13,917 | INFO |   Variant 'normalized': Нормализация: lowercase + удаление URL, пунктуации, чисел + пробелы
2026-03-22 03:20:22,544 | INFO |     Saved: data/prepared/text_normalized.csv
2026-03-22 03:20:22,545 | INFO |   Variant 'no_stopwords': Нормализация + удаление стоп-слов
2026-03-22 03:20:33,147 | INFO |     Saved: data/prepared/text_no_stopwords.csv
2026-03-22 03:20:33,148 | INFO |   Variant 'stemmed': Нормализация + удаление стоп-слов + стемминг
2026-03-22 03:20:49,299 | INFO |     Saved: data/prepared/text_stemmed.csv



  TEXT PREPROCESSING VARIANTS

  Sample (original, first 150 chars):
    "We just discovered another reason NOT to support the NFL The man who is the most anti-American person we know is now connected to The National Footbal..."

  raw: Без предобработки — исходный текст как есть
    Steps: 0
    Avg chars: 2463, Avg words: 405
    Sample: "We just discovered another reason NOT to support the NFL The man who is the most anti-American person we know is now connected to The National Footbal..."

  basic_clean: Базовая очистка: lowercase + удаление URL + лишние пробелы
    Steps: 3
    Avg chars: 2451, Avg words: 404
    Sample: "we just discovered another reason not to support the nfl the man who is the most anti-american person we know is now connected to the national footbal..."

  normalized: Нормализация: lowercase + удаление URL, пунктуации, чисел + пробелы
    Steps: 5
    Avg chars: 2375, Avg words: 399
    Sample: "we just discovered another reason not to support the nfl the man

In [142]:
def filter_trump(csv_path: str) -> str:
    df = pd.read_csv(csv_path)
    text_lower = df["text"].fillna("").str.lower()
    if "title" in df.columns:
        title_lower = df["title"].fillna("").str.lower()
    else:
        title_lower = pd.Series("", index=df.index)
    mask = text_lower.str.contains("trump", na=False) | title_lower.str.contains("trump", na=False)
    df_filtered = df[mask].reset_index(drop=True)
    p = Path(csv_path)
    out_path = p.parent / f"{p.stem}_trump{p.suffix}"
    df_filtered.to_csv(out_path, index=False)
    print(f"{len(df)} -> {len(df_filtered)} rows, saved: {out_path}")
    return str(out_path)
 

In [149]:
filter_trump("data/processed/test2.csv")

1954 -> 622 rows, saved: data/processed/test2_trump.csv


'data/processed/test2_trump.csv'